# 04 - Baseline and Enhanced Training

This notebook consolidates baseline mutation models, baseline HIV-ESM-2 classifier training, model comparison, and the rare-mutation-aware attention model.

GPU notebook: required for attention training; CPU is sufficient for many baseline classifier cells.


In [ ]:
# Colab/local project setup — see COLAB_SETUP.md
import sys
from pathlib import Path

_nb = Path.cwd().resolve()
_notebooks = _nb if (_nb / 'notebook_setup.py').exists() else _nb / 'notebooks'
if str(_notebooks) not in sys.path:
    sys.path.insert(0, str(_notebooks))

from notebook_setup import bootstrap_notebook_environment, export_notebook_globals

paths, pipeline_config = bootstrap_notebook_environment(
    install_deps=True,
    mount_drive=True,
)
globals().update(export_notebook_globals(paths, pipeline_config))

print('Project root:', PROJECT_ROOT)
print('Running in Colab:', IN_COLAB)
print('ENABLE_IMPROVED_PIPELINE:', ENABLE_IMPROVED_PIPELINE)



---

## Original Binary Mutation Baseline

Preserved from the original `02_baseline_development.ipynb`.


# Notebook 02: Baseline Model Development

## HIV Drug Resistance Prediction with ESM-2

---

**Objective**: Develop baseline model using binary mutation encoding.

**Approach**:
- Binary mutation encoding (1 if mutated from reference, 0 otherwise)
- XGBoost classifier per drug
- 5-fold stratified cross-validation

**Target**: Mean AUC ~0.955 (baseline for comparison)

---

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from data_processing import load_fasta, get_drug_list, load_unified_data
from feature_engineering import (
    create_binary_mutation_encoding,
    HIV_PROTEASE_REFERENCE, HIV_RT_REFERENCE
)
from models import train_xgboost, per_drug_training, aggregate_drug_results
from evaluation import compute_auc, stratified_cv, bootstrap_auc
from visualization import plot_roc_curves, plot_drug_comparison

# Random seed
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("Imports complete")

In [ ]:
# Project paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
RESULTS_DIR = PROJECT_ROOT / 'results'
FIGURES_DIR = PROJECT_ROOT / 'figures'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data directory: {DATA_DIR}")

## 1. Load Data

In [ ]:
# Load unified data
unified_data = load_unified_data(DATA_DIR)

print("\nLoaded data:")
for dc, data in unified_data.items():
    print(f"  {dc}: {len(data['sequences'])} sequences")

## 2. Create Binary Mutation Encoding

Encode each position as 1 (mutated from reference) or 0 (same as reference).

In [ ]:
# Reference sequences
references = {
    'PI': HIV_PROTEASE_REFERENCE,
    'NRTI': HIV_RT_REFERENCE,
    'NNRTI': HIV_RT_REFERENCE
}

print("Reference sequences:")
for dc, ref in references.items():
    print(f"  {dc}: {len(ref)} aa")

In [ ]:
# Create binary encodings
encodings = {}

for drug_class, data in unified_data.items():
    print(f"\nEncoding {drug_class}...")
    
    sequences = data['sequences']
    reference = references[drug_class]
    
    encoding = create_binary_mutation_encoding(sequences, reference)
    encodings[drug_class] = encoding
    
    # Statistics
    mutation_rate = encoding.mean(axis=0)
    print(f"  Shape: {encoding.shape}")
    print(f"  Mean mutations per sequence: {encoding.sum(axis=1).mean():.1f}")
    print(f"  Most mutated positions: {np.argsort(mutation_rate)[-5:][::-1] + 1}")

## 3. Train Baseline Models (XGBoost)

Train per-drug classifiers using XGBoost with 5-fold cross-validation.

In [ ]:
# Train baseline models for each drug class
baseline_results = {}

for drug_class, data in unified_data.items():
    print(f"\n{'='*60}")
    print(f"TRAINING {drug_class} BASELINE")
    print(f"{'='*60}")
    
    X = encodings[drug_class]
    phenotypes = data['phenotypes']
    drugs = get_drug_list(drug_class)
    
    results = per_drug_training(
        X, phenotypes, drugs,
        model_type='xgboost',
        n_splits=5,
        random_state=RANDOM_SEED
    )
    
    baseline_results[drug_class] = results

In [ ]:
# Diagnostic: list skipped drugs and reasons
for dc, results in baseline_results.items():
    skipped = {drug: res.get('reason') for drug, res in results.items() if isinstance(res, dict) and res.get('skipped', False)}
    if len(skipped) > 0:
        print(f"\n{dc} - skipped drugs:")
        for drug, reason in skipped.items():
            print(f"  {drug}: {reason}")
    else:
        print(f"\n{dc}: no skipped drugs")

## 4. Results Summary

In [ ]:
# Aggregate results
print("\n" + "="*60)
print("BASELINE RESULTS SUMMARY")
print("="*60)

all_aucs = []

for drug_class, results in baseline_results.items():
    print(f"\n{drug_class}:")
    summary = aggregate_drug_results(results)
    # print table if available
    cols = [c for c in ['drug','auc','n_samples'] if c in summary.columns]
    if not summary.empty and len(cols)>0:
        print(summary[cols].to_string(index=False))
    else:
        print('  No results for this drug class.')

    # collect AUCs safely
    for r in results.values():
        if isinstance(r, dict):
            a = r.get('auc', None)
        else:
            a = None
        if a is not None and not (isinstance(a, float) and np.isnan(a)):
            all_aucs.append(a)

# summary stats guarded against empty list
if len(all_aucs) == 0:
    print('\nNo valid AUC values to summarize.')
else:
    arr = np.array(all_aucs)
    print('\n' + '='*60)
    print(f"Overall Mean AUC: {arr.mean():.4f} (+/- {arr.std():.4f})")
    print(f"Median AUC: {np.median(arr):.4f}")
    print(f"Range: [{arr.min():.4f}, {arr.max():.4f}]")

In [ ]:
# Plot ROC curves for each drug class
for drug_class, results in baseline_results.items():
    fig = plot_roc_curves(
        results,
        title=f"Baseline ROC Curves - {drug_class}",
        save_path=FIGURES_DIR / f'baseline_roc_{drug_class}.png'
    )
    plt.show()

## 5. Save Baseline Results

In [ ]:
import pickle

# Save results for comparison with ESM-2
baseline_path = RESULTS_DIR / 'baseline_results.pkl'

with open(baseline_path, 'wb') as f:
    pickle.dump(baseline_results, f)

print(f"Saved baseline results to: {baseline_path}")

# Save as CSV for easy viewing
all_results = []
for drug_class, results in baseline_results.items():
    for drug, res in results.items():
        all_results.append({
            'drug_class': drug_class,
            'drug': drug,
            'auc': res['auc'],
            'n_samples': res['n_samples'],
            'n_resistant': res['n_resistant'],
            'n_susceptible': res['n_susceptible']
        })

results_df = pd.DataFrame(all_results)
results_df.to_csv(RESULTS_DIR / 'baseline_results.csv', index=False)

print(f"Saved baseline summary to: {RESULTS_DIR / 'baseline_results.csv'}")

## Summary

**Baseline Performance:**
- Expected mean AUC: ~0.955
- Using binary mutation encoding (position-wise)
- XGBoost classifier with 5-fold stratified CV

**Next steps:**
- Notebook 03: Extract ESM-2 embeddings
- Notebook 04: Compare ESM-2 approach to this baseline


---

## Original HIV-ESM-2 Classifier Training and Comparison

Preserved from the original `04_classification_evaluation.ipynb`.


# Notebook 04: Classification Evaluation

## HIV Drug Resistance Prediction with ESM-2

---

**Objective**: Train and evaluate classifiers using ESM-2 embeddings.

**Targets**:
- ESM-2 Mean AUC: 0.968
- Baseline (XGBoost) AUC: 0.955
- Improvement: +0.013 (p=0.0017)
- Drugs improved: 15/18

---

In [ ]:
import os
import sys
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from data_processing import load_unified_data, get_drug_list
from models import (
    per_drug_training, aggregate_drug_results, compare_models
)
from evaluation import (
    compute_auc, delong_test, bootstrap_auc,
    compare_esm2_vs_baseline
)
from visualization import plot_roc_curves, plot_drug_comparison

# Random seed
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("Imports complete")

In [ ]:
# Project paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
EMBEDDINGS_DIR = PROJECT_ROOT / 'data' / 'embeddings'
RESULTS_DIR = PROJECT_ROOT / 'results'
FIGURES_DIR = PROJECT_ROOT / 'figures'

print(f"Data: {DATA_DIR}")
print(f"Embeddings: {EMBEDDINGS_DIR}")

## 1. Load Data and Embeddings

In [ ]:
# Load unified data
unified_data = load_unified_data(DATA_DIR)

# Load ESM-2 embeddings
embeddings = {}
for drug_class in unified_data.keys():
    emb_path = EMBEDDINGS_DIR / f'{drug_class}_pooled_mean.npy'
    if emb_path.exists():
        embeddings[drug_class] = np.load(emb_path)
        print(f"{drug_class}: {embeddings[drug_class].shape}")
    else:
        print(f"{drug_class}: Embeddings not found!")

In [ ]:
# Load baseline results
baseline_path = RESULTS_DIR / 'baseline_results.pkl'

if baseline_path.exists():
    with open(baseline_path, 'rb') as f:
        baseline_results = pickle.load(f)
    print("Loaded baseline results")
else:
    print("Baseline results not found - run notebook 02 first")
    baseline_results = None

## 2. Train ESM-2 Classifiers

Compare multiple classifier types on ESM-2 embeddings.

In [ ]:
# Compare classifiers
classifier_types = ['logistic', 'xgboost', 'rf']

esm2_results = {}

for drug_class, data in unified_data.items():
    if drug_class not in embeddings:
        continue
        
    print(f"\n{'='*60}")
    print(f"{drug_class}")
    print(f"{'='*60}")
    
    X = embeddings[drug_class]
    phenotypes = data['phenotypes']
    drugs = get_drug_list(drug_class)
    
    esm2_results[drug_class] = {}
    
    for model_type in classifier_types:
        print(f"\n{model_type.upper()}:")
        results = per_drug_training(
            X, phenotypes, drugs,
            model_type=model_type,
            n_splits=5,
            random_state=RANDOM_SEED
        )
        esm2_results[drug_class][model_type] = results

## 3. Compare Classifier Performance

In [ ]:
# Summary by classifier type
print("\nClassifier Comparison (Mean AUC):")
print("-" * 50)

classifier_summary = []

for drug_class in esm2_results.keys():
    for model_type in classifier_types:
        results = esm2_results[drug_class][model_type]
        aucs = [r['auc'] for r in results.values()]
        
        classifier_summary.append({
            'drug_class': drug_class,
            'classifier': model_type,
            'mean_auc': np.mean(aucs),
            'std_auc': np.std(aucs)
        })

summary_df = pd.DataFrame(classifier_summary)
print(summary_df.pivot(index='drug_class', columns='classifier', values='mean_auc'))

In [ ]:
# Select best classifier (typically logistic regression for ESM-2)
BEST_CLASSIFIER = 'logistic'

# Create flattened results for comparison
esm2_flat = {}
for drug_class in esm2_results.keys():
    for drug, res in esm2_results[drug_class][BEST_CLASSIFIER].items():
        esm2_flat[drug] = res

## 4. ESM-2 vs Baseline Comparison

In [ ]:
# Compare ESM-2 to baseline
if baseline_results:
    # Flatten baseline results
    baseline_flat = {}
    for drug_class in baseline_results.keys():
        for drug, res in baseline_results[drug_class].items():
            baseline_flat[drug] = res
    
    # Get common drugs
    common_drugs = set(esm2_flat.keys()) & set(baseline_flat.keys())
    
    # Compare
    comparison_df = compare_esm2_vs_baseline(
        esm2_flat, baseline_flat, list(common_drugs)
    )
    
    print("\nESM-2 vs Baseline Comparison:")
    print(comparison_df.to_string(index=False))

In [ ]:
# Summary statistics
if baseline_results:
    esm2_aucs = [esm2_flat[d]['auc'] for d in common_drugs]
    baseline_aucs = [baseline_flat[d]['auc'] for d in common_drugs]
    
    print("\n" + "="*60)
    print("SUMMARY")
    print("="*60)
    print(f"\nESM-2 Mean AUC:    {np.mean(esm2_aucs):.4f} +/- {np.std(esm2_aucs):.4f}")
    print(f"Baseline Mean AUC: {np.mean(baseline_aucs):.4f} +/- {np.std(baseline_aucs):.4f}")
    print(f"Improvement:       {np.mean(esm2_aucs) - np.mean(baseline_aucs):.4f}")
    
    # Count drugs where ESM-2 is better
    n_improved = sum(1 for d in common_drugs if esm2_flat[d]['auc'] > baseline_flat[d]['auc'])
    print(f"\nDrugs improved: {n_improved}/{len(common_drugs)}")
    
    # Paired test
    from scipy.stats import wilcoxon
    stat, p_value = wilcoxon(esm2_aucs, baseline_aucs)
    print(f"Wilcoxon p-value: {p_value:.4f}")

## 5. Visualization

In [ ]:
# Bar chart comparison
if baseline_results:
    fig = plot_drug_comparison(
        esm2_flat, baseline_flat,
        title="ESM-2 vs Baseline AUC by Drug",
        save_path=FIGURES_DIR / 'esm2_vs_baseline_comparison.png'
    )
    plt.show()

In [ ]:
# ROC curves for ESM-2
for drug_class in esm2_results.keys():
    results = esm2_results[drug_class][BEST_CLASSIFIER]
    
    fig = plot_roc_curves(
        results,
        title=f"ESM-2 ROC Curves - {drug_class}",
        save_path=FIGURES_DIR / f'esm2_roc_{drug_class}.png'
    )
    plt.show()

## 6. Save Results

In [ ]:
# Save ESM-2 results
esm2_path = RESULTS_DIR / 'esm2_results.pkl'
with open(esm2_path, 'wb') as f:
    pickle.dump(esm2_results, f)
print(f"Saved ESM-2 results to: {esm2_path}")

# Save comparison
if baseline_results:
    comparison_df.to_csv(RESULTS_DIR / 'esm2_vs_baseline.csv', index=False)
    print(f"Saved comparison to: {RESULTS_DIR / 'esm2_vs_baseline.csv'}")

## Summary

**Expected results:**
- ESM-2 Mean AUC: ~0.968
- Baseline Mean AUC: ~0.955
- Improvement: ~+0.013
- Drugs improved: 15/18

**Key findings:**
- Logistic regression works well with ESM-2 embeddings
- Consistent improvement across drug classes
- Statistical significance confirmed with paired test

**Next steps:**
- Notebook 05: Interpretability analysis
- Notebook 06: External validation


---

## Rare-Mutation-Aware Attention Training

This replaces the separate rare-mutation attention notebook and uses outputs from notebooks 02 and 03.


In [ ]:
# Rare-mutation-aware attention model training
RUN_ATTENTION_TRAINING = pipeline_config.RUN_ATTENTION_TRAINING
ATTENTION_EPOCHS = 10
ATTENTION_DRUG_CLASS = 'PI'   # PI, NRTI, or NNRTI
ATTENTION_DRUG = None         # None selects the first available drug in the class

if RUN_ATTENTION_TRAINING:
    import pickle
    import numpy as np
    import torch
    from src.data_processing import load_unified_data
    from src.models import train_attention_model

    unified_data = load_unified_data(PROCESSED_DIR)
    data = unified_data[ATTENTION_DRUG_CLASS]
    drug = ATTENTION_DRUG or data['drugs'][0]
    label_col = f'{drug}_class2'
    if label_col not in data['phenotypes'].columns:
        raise ValueError(f'Missing binary label column: {label_col}')

    embedding_candidates = [
        EMBEDDINGS_DIR / ATTENTION_DRUG_CLASS / 'per_residue.npy',
        EMBEDDINGS_DIR / f'{ATTENTION_DRUG_CLASS}_per_residue.npy',
    ]
    embedding_path = next((p for p in embedding_candidates if p.exists()), None)
    if embedding_path is None:
        raise FileNotFoundError('Per-residue embeddings not found. Run notebook 03 per-residue section first.')

    embeddings = np.load(embedding_path, allow_pickle=True)
    labels_raw = data['phenotypes'][label_col].values
    valid = ~np.isnan(labels_raw)
    labels = labels_raw[valid].astype(int)
    embeddings_valid = [np.asarray(embeddings[i], dtype=np.float32) for i in range(len(embeddings)) if valid[i]]

    weights_path = PROJECT_ROOT / 'data' / 'rare_mutations' / ATTENTION_DRUG_CLASS / 'normalized_weights.npy'
    rare_weights = None
    if weights_path.exists():
        weights = np.load(weights_path, allow_pickle=True)
        rare_weights = [np.asarray(weights[i], dtype=np.float32) for i in range(len(weights)) if valid[i]]
        print('Loaded normalized rare mutation weights:', weights_path)
    else:
        print('Rare mutation weights not found; training attention model without rarity modulation.')

    model = train_attention_model(
        embeddings_valid,
        labels,
        rare_mutation_weights_list=rare_weights,
        epochs=ATTENTION_EPOCHS,
        verbose=True,
    )

    out_dir = RESULTS_DIR / 'attention_models'
    out_dir.mkdir(parents=True, exist_ok=True)
    model_path = out_dir / f'{ATTENTION_DRUG_CLASS}_{drug}_attention_model.pt'
    torch.save(model.state_dict(), model_path)
    print('Saved attention model:', model_path)
else:
    print('Set RUN_ATTENTION_TRAINING = True to train the rare-mutation-aware attention model.')


## Workflow Handoff

Next notebook: `05_evaluation_and_validation.ipynb`.

When `ENABLE_IMPROVED_PIPELINE = True`, the rare-mutation-aware attention model is trained here before ternary/calibration evaluation in notebook 05.
